# Split-Hygiene Audit (CLASSIFIER)

**Hard-fails** with `AssertionError` if any of the following is wrong:

1. **Subject-overlap** — a single `Pseudonym` is found in more than one of the train/val/test split CSVs.
2. **Duplicate matrices** — two distinct `.npz` files contain the exact same array (upstream preprocessing bug).
3. **StratifiedGroupKFold** — re-running CV with the saved seed produces overlapping validation folds.
4. **GAAE ↔ downstream cohort policy** — the chosen policy (`shared` or `disjoint`) is asserted.

Wire this notebook's checks into every production notebook via `from common.sanity import run_full_audit`.

In [ ]:
# === Papermill parameters (injected by run_experiment.py) ===
# Safe interactive defaults: None keeps the original Jupyter behaviour
# (interactive checkpoint/threshold prompts, JSON-config loading).
EXPERIMENT_ID = None
MODE = None
MODEL = None
DATASET = None
SEED = 42
GAAE_CHECKPOINT_PATH = None   # None -> interactive checkpoint picker
THRESHOLD_MODE = None         # None -> interactive prompt; else youden | best-f1 | fixed
FIXED_THRESHOLD = None        # required when THRESHOLD_MODE is fixed
WANDB_ENABLED = True          # W&B logging is on by default
OUTPUT_DIR = None             # defaults to outputs/<experiment-id>/ when run standalone
RESOLVED_CONFIG = None        # merged hyperparameter dict; overrides on-disk JSON when set
RUN_DIR = None                # set by the runner: where run_summary.json / artifacts go
RUN_NAME = None               # set by the runner: the W&B run name


In [ ]:
import sys
from pathlib import Path
import os
import pandas as pd

V2_ROOT = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER')
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

from common.sanity import (
    assert_splits_clean,
    assert_no_duplicate_matrices,
    audit_groupkfold,
    assert_cohort_policy,
)
print('Sanity utilities loaded')

## Configure paths

Edit the constants below if you ship a new data version (`__fc_dmn_sch200_flat__` → `__fc_hippo_tian2_flat__`, etc.). Everything downstream picks them up automatically.

In [ ]:
import sys
if '/mnt/e/fyassine/ad-early-detection' not in sys.path:
    sys.path.insert(0, '/mnt/e/fyassine/ad-early-detection')
from DATA.src.splitting.load_splits import splits_dir, split_csv_paths

DATA_VERSION   = '__fc_wholebrain_sch200_flat__'
DATA_ROOT      = Path('/mnt/e/fyassine/ad-early-detection/DATA/DELCODE') / DATA_VERSION
MATRICES_DIR   = DATA_ROOT / 'matrices'
METADATA_DIR   = DATA_ROOT / 'metadata'
COHORTS_CSV    = METADATA_DIR / 'cohorts.csv'

# Pretrain split (all cohorts) — used to pretrain the GAAE encoder.
SPLITS_DIR     = splits_dir('pretrain')
SPLIT_CSVS = {
    'train': str(SPLITS_DIR / 'train.csv'),
    'val':   str(SPLITS_DIR / 'val.csv'),
    'test':  str(SPLITS_DIR / 'test.csv'),
}

# Downstream split (mci/converter only) — used by every classifier built on
# top of the frozen GAAE encoder (GEC, GELSTM, Long-GEC-MLP, LogReg, PROGNOSER).
SPLITS_DIR_DOWNSTREAM = splits_dir('downstream')
SPLIT_CSVS_DOWNSTREAM = {
    'train': str(SPLITS_DIR_DOWNSTREAM / 'train.csv'),
    'val':   str(SPLITS_DIR_DOWNSTREAM / 'val.csv'),
    'test':  str(SPLITS_DIR_DOWNSTREAM / 'test.csv'),
}

for label, mapping in [('pretrain', SPLIT_CSVS), ('downstream', SPLIT_CSVS_DOWNSTREAM)]:
    for name, p in mapping.items():
        exists = os.path.exists(p)
        print(f'  [{label}] {name:<6} {p}  {"OK" if exists else "MISSING"}')

## 1. Subject-overlap audit

For every pair of split CSVs, assert that the sets of `Pseudonym` values are disjoint. Raises `AssertionError` listing the offenders on any overlap.

In [ ]:
rep_pretrain = assert_splits_clean(SPLIT_CSVS, id_col='Pseudonym', raise_on_overlap=True)
print('Pretrain split sizes (subjects):', rep_pretrain['sizes'])
print('Pairwise-disjoint:', rep_pretrain['clean'])

In [ ]:
if all(os.path.exists(p) for p in SPLIT_CSVS_DOWNSTREAM.values()):
    rep_downstream = assert_splits_clean(SPLIT_CSVS_DOWNSTREAM, id_col='Pseudonym', raise_on_overlap=True)
    print('Downstream split sizes (subjects):', rep_downstream['sizes'])
    print('Pairwise-disjoint:', rep_downstream['clean'])
else:
    print('Downstream splits not present at expected paths — skipping')

## 2. Duplicate-matrix audit

Content-hashes every `.npz` correlation matrix referenced by the production splits. If two files hash to the same value, an upstream preprocessing step produced duplicate scans — these contaminate any train/test boundary regardless of split hygiene.

In [ ]:
import glob

all_npz = sorted(glob.glob(str(MATRICES_DIR / '*_z_transformed.npz')))
print(f'Found {len(all_npz)} .npz files in {MATRICES_DIR}')

dup_rep = assert_no_duplicate_matrices(all_npz, raise_on_dup=False)
print(f'  unique content hashes: {dup_rep["n_unique"]} / {dup_rep["n_files"]}')
if dup_rep['clean']:
    print('  Clean: no two files share content.')
else:
    print(f'  Found {len(dup_rep["duplicates"])} duplicate group(s):')
    for d in dup_rep['duplicates'][:5]:
        print(f'    hash={d["hash"][:10]}  paths={d["paths"]}')
    raise AssertionError('Duplicate matrices found — see list above')

## 3. StratifiedGroupKFold re-check

Rebuild the CV pool the way the production notebooks do (train + val merged), then re-run `StratifiedGroupKFold(n_splits=5, seed=42)` and assert that no subject appears in two validation folds.

In [ ]:
train_df = pd.read_csv(SPLIT_CSVS_DOWNSTREAM['train'])
val_df   = pd.read_csv(SPLIT_CSVS_DOWNSTREAM['val'])
cv_pool  = pd.concat([train_df, val_df], ignore_index=True)
cv_pool['Pseudonym'] = cv_pool['Pseudonym'].astype(str)

subjects = cv_pool['Pseudonym'].tolist()
labels   = (cv_pool['diagnosis'] == 'converter').astype(int).tolist()
print(f'CV pool: {len(subjects)} subjects ({sum(labels)} converter, {len(subjects)-sum(labels)} stable)')

kf_rep = audit_groupkfold(subjects, labels, n_splits=5, seed=42)
print('Per-fold sizes:', [(f["n_train"], f["n_val"]) for f in kf_rep['folds']])
print('Pairwise-disjoint val folds:', kf_rep['clean'])

## 4. Cohort policy (GAAE pretrain ↔ downstream classifier)

State the policy explicitly. Two valid choices:

* `policy='shared'` — GAAE was pretrained **on the same subjects** later used downstream. Acceptable because GAAE is unsupervised, but you must say so.
* `policy='disjoint'` — GAAE pretrained on subjects never seen at classification time. Stricter, eliminates any unsupervised-representation leakage.

Whichever you pick, it goes on the methods page of the paper.

In [ ]:
# Edit this to match the project's actual choice.
COHORT_POLICY = 'shared'

gaae_train = pd.read_csv(SPLIT_CSVS['train'])
gaae_subjects = gaae_train['Pseudonym'].astype(str).tolist()
downstream_subjects = subjects   # MCI/converter pool from cell above

policy_rep = assert_cohort_policy(
    gaae_pretrain_subjects=gaae_subjects,
    downstream_subjects=downstream_subjects,
    policy=COHORT_POLICY,
)
print(policy_rep)

## Summary

If every cell above ran without `AssertionError`, splits are clean. Run this notebook end-to-end whenever you regenerate splits or change data versions.